# WalletShare Alternatives — Monthly Campaign Pipeline

Enriches WalletShare metrics with Salesforce firm/contact attributes and applies quintile rankings.

**Parameters:**
- `catalog` — Target catalog
- `bronze_ws_schema` — Bronze WalletShare schema
- `bronze_sf_schema` — Bronze Salesforce schema
- `silver_schema` — Silver output schema

**Silver Layer:**
- `pwfirm_attributes` — Firm attributes from Salesforce
- `campaign_vehicle_1031_oz` — Filtered Real Estate DST 1031/OZ with quintile
- `campaign_overall` — Enriched overall with quintile rankings

In [0]:
# Fetch all parameters separately
catalog = dbutils.widgets.get("catalog")
silver_sf_schema = dbutils.widgets.get("silver_sf_schema")
silver_schema = dbutils.widgets.get("silver_schema")
bronze_ws_schema = dbutils.widgets.get("bronze_ws_schema")

bronze_vehicle = f"{catalog}.{bronze_ws_schema}.ws_alternatives_vehicle"

In [0]:
# Auto-detect as_of_date: always process the latest available date from bronze
from pyspark.sql import functions as F


max_bronze = spark.table(bronze_vehicle).agg(F.max("as_of_date")).collect()[0][0]

max_date = str(max_bronze)
dbutils.widgets.text("max_date", max_date)
print(f"Processing max_date: {max_date}")

Processing max_date: 2026-07-31


In [0]:
%sql
-- Silver: PWFirm Attributes (snapshot per run date — preserves point-in-time history)

DELETE FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.pwfirm_attributes')
WHERE as_of_date = CAST(:max_date AS DATE);

-- Insert fresh snapshot
INSERT INTO IDENTIFIER(:catalog || '.' || :silver_schema || '.pwfirm_attributes')
WITH current_accounts AS (
  -- Only the currently-valid SCD2 version of each Account record
  SELECT *
  FROM IDENTIFIER(:catalog || '.' || :silver_sf_schema || '.account')
  WHERE __END_AT IS NULL
    AND Firm_CRD__c IS NOT NULL
),
deduped AS (
  -- Safety net: if a Firm_CRD__c still maps to more than one current
  -- Account record, keep exactly one, preferring the most recently modified.
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY Firm_CRD__c
      ORDER BY LastModifiedDate DESC, Id
    ) AS rn
  FROM current_accounts
)
SELECT
  a.Firm_CRD__c                            AS firm_crd,
  a.Name                                   AS institution_name,
  rt.Name                                  AS institution_record_type,
  a.X18_Digit_Institution_ID__c            AS institution_id_18digit,
  a.Funds_Available__c                     AS funds_available,
  ROUND(a.Total_Assets_mil__c, 2)          AS `total_assets_$mil`,
  ROUND(a.Total_Fund_Commitment_new__c, 2) AS `total_fund_commitments_($mn)`,
  a.Last_Activity_Date__c                  AS last_activity,
  a.Last_Appointment_Date__c               AS last_appointment_date,
  a.OwnerId                                AS lead,
  a.Internal_Coverage__c                   AS support,
  ROUND(a.AUM__c, 2)                       AS `aum_$mil`,
  a.Lead_Management__c                     AS lead_management,
  a.Relationship_Status__c                 AS relationship_status,
  a.Likelihood_of_Reinvestment__c          AS likelihood_of_reinvestment,
  a.BillingCity                            AS business_city,
  a.BillingState                           AS business_state_province,
  a.BillingCountry                         AS business_country,
  a.Phone                                  AS phone,
  a.Preqin_Firm_ID__c                      AS preqin_firm_id,
  a.ParentId                               AS parent_institution,
  a.Region__c                              AS sub_region,
  a.Territory__c                           AS territory,
  CAST(:max_date AS DATE)                  AS as_of_date,
  current_timestamp()                      AS loaded_at

FROM deduped a
LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_sf_schema || '.recordtype') rt
  ON a.RecordTypeId = rt.Id
WHERE a.rn = 1;

num_affected_rows,num_inserted_rows
48233,48233


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_contact_identity AS
WITH current_contacts AS (
  -- Only the currently-valid SCD2 version of each Contact record
  SELECT *
  FROM IDENTIFIER(:catalog || '.' || :silver_sf_schema || '.contact')
  WHERE __END_AT IS NULL
    AND Rep_CRD__c IS NOT NULL
),
current_accounts AS (
  -- Only the currently-valid SCD2 version of each Account record
  SELECT Id, Funds_Available__c
  FROM IDENTIFIER(:catalog || '.' || :silver_sf_schema || '.account')
  WHERE __END_AT IS NULL
),
matched AS (
  SELECT
    c.Rep_CRD__c              AS crd_number,
    c.X18_Digit_Contact_ID__c AS sf_18_digit_id,
    c.LastModifiedDate
  FROM current_contacts c
  LEFT JOIN current_accounts a
    ON c.AccountId = a.Id
  WHERE a.Funds_Available__c LIKE '%OZ%'
     OR a.Funds_Available__c LIKE '%All Funds%'
),
deduped AS (
  -- Safety net: if a Rep_CRD__c still maps to more than one current
  -- Contact record, keep exactly one, preferring the most recently modified.
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY crd_number
      ORDER BY LastModifiedDate DESC
    ) AS rn
  FROM matched
)
SELECT crd_number, sf_18_digit_id
FROM deduped
WHERE rn = 1;

In [0]:
%sql
-- Silver: Campaign Vehicle 1031/OZ (snapshot per max_date)

DELETE FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.campaign_vehicle_1031_oz')
WHERE as_of_date = :max_date;

INSERT INTO IDENTIFIER(:catalog || '.' || :silver_schema || '.campaign_vehicle_1031_oz')
WITH ws_vehicle_current AS (
  SELECT
    as_of_date, contact_id, crd_number, asset_class, vehicle_wrapper,
    sales_rank_90day, sales_score_90day, sales_rank_180day, sales_score_180day,
    sales_rank_12mo, sales_score_12mo, sales_12mo, sales_frequency,
    COALESCE(try_cast(last_purchase AS DATE), try_to_date(last_purchase, 'M/d/yyyy')) AS last_purchase,
    max_purchase, new_manager, rank_aum, aum_score, aum,
    reds_rank_90day, reds_score_90day, reds_rank_180day, reds_score_180day,
    reds_rank_12mo, reds_score_12mo, reds_12mo, reds_frequency,
    COALESCE(try_cast(last_red AS DATE), try_to_date(last_red, 'M/d/yyyy')) AS last_red,
    max_red, sales_growth_yr, sales_momentum_score
  FROM IDENTIFIER(:catalog || '.' || :bronze_ws_schema || '.ws_alternatives_vehicle')
  WHERE as_of_date = CAST(:max_date AS DATE)
),
filtered AS (
  SELECT *
  FROM ws_vehicle_current
  WHERE asset_class = 'Real Estate'
    AND vehicle_wrapper = 'DST 1031 - Opportunity Zone Fund'
),
ranked AS (
  SELECT *,
    NTILE(5) OVER (
      ORDER BY
        CASE WHEN sales_rank_12mo IS NOT NULL THEN 0 ELSE 1 END ASC,
        sales_rank_12mo ASC,
        rank_aum ASC NULLS LAST
    ) AS raw_bucket
  FROM filtered
)
SELECT * EXCEPT (raw_bucket), 6 - raw_bucket AS `1031/oz_quintile`,
  current_timestamp() AS loaded_at
FROM ranked;

num_affected_rows,num_inserted_rows
6561,6561


In [0]:
%sql
-- Silver: Campaign Overall (snapshot per max_date)

DELETE FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.campaign_overall')
WHERE as_of_date = CAST(:max_date AS DATE);

INSERT INTO IDENTIFIER(:catalog || '.' || :silver_schema || '.campaign_overall')
WITH ws_overall_current AS (
  SELECT *
  FROM IDENTIFIER(:catalog || '.' || :bronze_ws_schema || '.ws_alternatives_overall')
  WHERE as_of_date = try_cast(:max_date AS DATE)
),
ws_overall_enriched AS (
  SELECT
    o.crd_number, o.contact_id, o.firm_crd, o.as_of_date, o.firm_id, o.office_id,
    CAST(NULL AS STRING) AS sf_18_digit_id_formula,
    cl.sf_18_digit_id,
    o.firm_name,
    pw.funds_available,
    pw.`total_fund_commitments_($mn)`  AS firm_total_commitments_mn,
    pw.last_activity                   AS Firm_Last_Activity,
    pw.last_appointment_date           AS Firm_Last_Appointment,
    pw.institution_record_type         AS Institution_Type,
    o.channel_name, o.territory_name, o.rep_type, o.contact_name,
    o.email_address, o.office_phone, o.address_line_1, o.address_line_2,
    o.city, o.state, o.zip, o.relationship,
    o.sales_rank_90day, o.sales_score_90day, o.sales_rank_180day, o.sales_score_180day,
    o.sales_rank_12mo, o.sales_score_12mo, o.sales_12mo, o.sales_frequency,
    o.last_purchase, o.max_purchase, o.new_manager, o.qualified_investor,
    o.rank_aum, o.aum_score, o.aum,
    o.reds_rank_90day, o.reds_score_90day, o.reds_rank_180day, o.reds_score_180day,
    o.reds_rank_12mo, o.reds_score_12mo, o.reds_12mo, o.reds_frequency,
    o.last_red, o.max_red, o.sales_growth_yr, o.sales_momentum_score,
    v.asset_class    ,
    v.vehicle_wrapper    ,
    CASE
      WHEN v.crd_number IS NOT NULL AND v.sales_rank_12mo IS NULL THEN 0
      ELSE v.sales_rank_12mo
    END AS `1031/oz_sales_rank_12mo`,
    v.`1031/oz_quintile`
  FROM ws_overall_current o
  LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_schema || '.pwfirm_attributes') pw
    ON try_cast(o.firm_crd AS BIGINT) = try_cast(pw.firm_crd AS BIGINT)
    AND pw.as_of_date = try_cast(:max_date AS DATE)
  LEFT JOIN vw_contact_identity cl
    ON try_cast(o.crd_number AS BIGINT) = try_cast(cl.crd_number AS BIGINT)
  LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_schema || '.campaign_vehicle_1031_oz') v
    ON try_cast(o.crd_number AS BIGINT) = try_cast(v.crd_number AS BIGINT)
    AND v.as_of_date = try_cast(:max_date AS DATE)
  -- Exclude house/placeholder accounts (e.g. "House Account - General",
  -- "House Tiedemann Advisors Llc"). These are firm-level umbrella
  -- accounts, not real advisors, and carry large aggregated AUM that
  -- distorts totals and quintile rankings. Identified by contact_name
  -- only, NOT by crd_number — confirmed 345 real, named advisors also
  -- have a null crd_number for unrelated reasons and must not be
  -- excluded by this filter.
  WHERE o.contact_name IS NULL OR o.contact_name NOT LIKE 'House%'
),
flagged AS (
  SELECT *,
    CASE WHEN rank_aum IS NOT NULL THEN 'Y' ELSE '' END AS alt_user
  FROM ws_overall_enriched
),
ranked AS (
  SELECT *,
    NTILE(5) OVER (
      PARTITION BY (
        funds_available IS NOT NULL
        AND funds_available != '#N/A'
        AND rank_aum IS NOT NULL
      )
      ORDER BY rank_aum ASC
    ) AS overall_bucket,
    NTILE(5) OVER (
      PARTITION BY (`1031/oz_sales_rank_12mo` IS NOT NULL)
      ORDER BY `1031/oz_sales_rank_12mo` ASC
    ) AS oz_bucket
  FROM flagged
)
SELECT
  crd_number, 
  contact_id, 
  firm_crd,
  as_of_date, 
  firm_id, 
  office_id,
  sf_18_digit_id_formula, 
  sf_18_digit_id, 
  firm_name, 
  funds_available,
  firm_total_commitments_mn, 
  Firm_Last_Activity, 
  Firm_Last_Appointment,
  Institution_Type, 
  channel_name, 
  territory_name, 
  rep_type,
  contact_name,
  email_address, 
  office_phone,
  address_line_1, address_line_2, city, state, zip,
  relationship, sales_rank_90day, sales_score_90day, sales_rank_180day, sales_score_180day,
  sales_rank_12mo, sales_score_12mo, sales_12mo, sales_frequency, last_purchase,
  max_purchase, new_manager, qualified_investor, rank_aum, alt_user, aum_score,
  CASE
    WHEN funds_available IS NOT NULL
     AND funds_available != '#N/A'
     AND rank_aum IS NOT NULL
    THEN 6 - overall_bucket
    ELSE NULL
  END AS overall_alt_quintile,
  aum, reds_rank_90day, reds_score_90day, reds_rank_180day, reds_score_180day,
  reds_rank_12mo, reds_score_12mo, reds_12mo, reds_frequency, last_red, max_red,
  sales_growth_yr, sales_momentum_score, asset_class, vehicle_wrapper,
  `1031/oz_sales_rank_12mo`,
  CASE
    WHEN `1031/oz_sales_rank_12mo` IS NOT NULL THEN 6 - oz_bucket
    ELSE NULL
  END AS `1031/oz_quintile`,
  current_timestamp() AS loaded_at
FROM ranked;

num_affected_rows,num_inserted_rows
96441,96441
